# 05. 입찰메이트 RFP RAG — Phase 4 근거 기반 답변·요약

유영관 · 2026-11-11

Phase 3의 원문·청크·인덱스를 그대로 이어받아 OpenAI 답변 생성 인터페이스를 연결합니다.
답변의 각 주장에는 실제 검색 근거의 인용문이 필요하고, 원문 문자 범위로 돌아갈 수 있는지 검사합니다.

| 기능 | Phase 4에서 확인할 내용 |
| --- | --- |
| 사실 Q&A | 검색 → 구조화된 주장 생성 → 원문 인용 검사 |
| 전체 문서 요약 | 문서의 모든 청크를 배치에 배정 → 각 배치 요약 → 검증된 주장 집계 |
| 문서 비교 | 문서별 근거를 확보하고 문서별 주장으로 비교 |
| 후속 질문 | 선택 문서와 사용자 질문만 유지하며 다시 검색 |
| 근거 부족 처리 | 확인하지 못한 항목과 일부 답변 상태를 표시 |

**기본 Run All은 외부 API를 호출하지 않습니다.** 아래 기본 셀은 실제 원문·검색 결과·요약 실행 계획만 표시합니다.
실제 답변·요약·의미 검색 성능은 사용자가 로컬에서 해당 옵션을 켜고 실행한 뒤에 측정할 수 있습니다.
모의 모델 응답을 실제 OpenAI 답변처럼 표시하지 않습니다. GitHub나 Drive를 변경하는 작업은 없습니다.


## 1. 먼저 노트북 Python과 저장 인덱스의 버전 확인

Mac에서 프로젝트 폴더의 터미널을 열고 README의 `.venv/bin/python` 명령으로 설치하세요.
VS Code에서는 노트북 오른쪽 위 커널 → **Select Another Kernel → Python Environments**에서
이번 프로젝트의 `.venv/bin/python`을 선택하거나 등록한 `Python (RFP Phase 4)` 커널을 선택합니다.
터미널에서 가상환경을 활성화한 것과 노트북의 커널 선택은 별개입니다.

첫 셀은 실제 실행 경로와 numpy/scipy/scikit-learn 버전을 출력합니다.
버전이 다르면 검색 모듈을 불러오기 전에 멈추고 정확한 설치 명령을 안내합니다.
인덱스와 기존 평가 보고서를 자동으로 다시 만들거나 덮어쓰지 않습니다.


In [15]:
%pip install "openai>=3,<4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 44.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [openai] 9/10 [openai]c]
Note: you may need to restart the kernel to use updated packages.


In [16]:
import sys
import openai
from openai import OpenAI

print("Python:", sys.executable)
print("OpenAI SDK:", openai.__version__)
print("설치 위치:", openai.__file__)

Python: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase3_local_20261111/.venv/bin/python
OpenAI SDK: 3.49.0
설치 위치: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase3_local_20261111/.venv/lib/python3.11/site-packages/openai/__init__.py


In [18]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from collections import Counter
import json
import os
import sys

PROJECT_ROOT = None  # 다른 위치에서 열었으면 압축을 푼 Phase 4 폴더의 Path를 지정
ENABLE_OPENAI = True
RUN_LIVE_QA = True
RUN_LIVE_SUMMARY = False
RUN_DEV_EVALUATION = False
RUN_SEMANTIC_COMPARISON = False
DEV_CASE_LIMIT = 5
RUN_OUTPUT_DIR = None  # 실 API 옵션이 켜진 실행에서만 새 폴더를 생성
RESULT_SEQUENCE = 0

if PROJECT_ROOT is None:
    candidates = [Path.cwd(), Path.cwd().parent,
                  Path.cwd() / 'rfp_phase4_local_20261111',
                  Path.cwd() / 'output/rfp_phase4_local_20261111']
    ROOT = next((path.resolve() for path in candidates
                 if (path / 'rfp_phase4/runtime.py').is_file()
                 and (path / 'index/index_meta.json').is_file()), None)
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if ROOT is None or not (ROOT / 'rfp_phase4/runtime.py').is_file():
    raise FileNotFoundError('PROJECT_ROOT에 압축을 푼 Phase 4 패키지 폴더를 지정하세요.')

print('실행 중인 Python:', sys.executable)
print('Python 버전:', sys.version.split()[0])
print('프로젝트 경로:', ROOT)
saved_versions = json.loads((ROOT / 'index/index_meta.json').read_text(encoding='utf-8'))['dependencies']
version_mismatches = []
runtime_dependency_versions = {}
for package in ['numpy', 'scipy', 'scikit-learn']:
    try:
        current = version(package)
    except PackageNotFoundError:
        current = '설치되지 않음'
    expected = saved_versions[package]
    runtime_dependency_versions[package] = current
    matches = current == expected
    print(f'{package}: 저장된 버전={expected}, 현재 버전={current}, 일치={matches}')
    if not matches:
        version_mismatches.append(package)
if version_mismatches:
    raise RuntimeError(
        '노트북 커널과 저장 인덱스의 의존성 버전이 다릅니다. '
        'VS Code 오른쪽 위 커널에서 프로젝트 .venv/bin/python을 선택하세요. '
        '프로젝트 터미널 설치 명령: ./.venv/bin/python -m pip install -r requirements-phase4.txt '
        '설치 후 커널을 Restart하고 첫 셀부터 실행하세요. OpenAI 옵션 때문인 오류가 아닙니다.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
live_options = [RUN_LIVE_QA, RUN_LIVE_SUMMARY, RUN_DEV_EVALUATION, RUN_SEMANTIC_COMPARISON]
if any(live_options) and not ENABLE_OPENAI:
    raise ValueError('실 API 실행 옵션을 사용할 때는 ENABLE_OPENAI도 True로 설정하세요.')
print('환경 버전 검사: 통과')
print('OpenAI 활성화:', ENABLE_OPENAI, '/ 실제 실행 옵션:', any(live_options))
print('기본 Run All: 원문·검색·계획 점검, API 호출 0회')


실행 중인 Python: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase3_local_20261111/.venv/bin/python
Python 버전: 3.11.7
프로젝트 경로: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase4_local_20261111
numpy: 저장된 버전=2.3.5, 현재 버전=2.3.5, 일치=True
scipy: 저장된 버전=1.17.0, 현재 버전=1.17.0, 일치=True
scikit-learn: 저장된 버전=1.8.0, 현재 버전=1.8.0, 일치=True
환경 버전 검사: 통과
OpenAI 활성화: True / 실제 실행 옵션: True
기본 Run All: 원문·검색·계획 점검, API 호출 0회


## 2. Phase 3 결과를 변경하지 않고 불러오기

기존 원문·청크·검색 인덱스의 버전과 무결성을 검사합니다.
Phase 4 설정과 출력은 `configs/phase4.json`, `phase4_outputs/`, `phase4_reports/`를 사용합니다.
Phase 3의 설정·동결 결과·기존 holdout 결과는 비교 기준으로 보존합니다.

원문 인용 검사는 이미 파싱한 텍스트와 정확히 일치하는지 확인합니다.
인용문이 주장의 의미를 실제로 뒷받침하는지, 모든 PDF 그림과 표를 이해했는지는 별도 평가 대상입니다.


In [19]:
from rfp_phase4.runtime import bootstrap
from rfp_phase4.rag import AnswerSession
from rfp_phase4.llm import LLMConfig, OpenAIGroundedLLM, PROMPT_VERSION
from rfp_phase4.summarization import FullDocumentSummarizer
from rfp_phase4.costs import estimate_usage_cost
from local_rag.io import read_jsonl

app = bootstrap(ROOT)
index = app['index']
documents = app['documents']
chunks = app['chunks']
registry = app['registry']
config = app['config']
source_by_id = {document['doc_id']: document for document in documents}
catalog = index.get_catalog()
catalog_by_id = {row['doc_id']: row for row in catalog}

def show_json(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))

def save_run_json(stem, value):
    if RUN_OUTPUT_DIR is None:
        return None
    encoded = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + '\n'
    number = 1
    while True:
        suffix = '' if number == 1 else f'_{number:03d}'
        path = RUN_OUTPUT_DIR / f'{stem}{suffix}.json'
        try:
            with path.open('x', encoding='utf-8') as handle:
                handle.write(encoded)
            return path
        except FileExistsError:
            number += 1

def new_run_subdirectory(stem):
    if RUN_OUTPUT_DIR is None:
        raise RuntimeError('API 연결 셀을 먼저 실행해 이번 실험의 출력 폴더를 만드세요.')
    number = 1
    while True:
        suffix = '' if number == 1 else f'_{number:03d}'
        path = RUN_OUTPUT_DIR / f'{stem}{suffix}'
        try:
            path.mkdir()
            return path
        except FileExistsError:
            number += 1

def show_result(result):
    global RESULT_SEQUENCE
    text = result.get('markdown')
    if text is None:
        text = registry.render(result.get('validation', result))
    print(text)
    if result.get('error_detail'):
        print('생성 실패 원인:', result['error_detail'])
    show_json({key: result.get(key) for key in
               ['status', 'execution_status', 'answer_generated', 'input_chunk_coverage',
                'validated_chunk_coverage', 'semantic_coverage_verified'] if key in result})
    validation_detail = result.get('validation') or {}
    if validation_detail.get('errors'):
        print('출처 검증 세부 오류:')
        show_json(validation_detail['errors'])
        print('검증되지 않은 인용 데이터:')
        show_json([row for row in validation_detail.get('citation_diagnostics', [])
                   if row.get('outcome') not in {'already_exact', 'whitespace_only'}])
    recovery = validation_detail.get('citation_recovery') or {}
    if recovery.get('corrected_citations'):
        print('공백 차이를 원문으로 복원한 인용 수:', recovery['corrected_citations'])
    if RUN_OUTPUT_DIR is not None:
        RESULT_SEQUENCE += 1
        saved_result = save_run_json(f'result_{RESULT_SEQUENCE:03d}', result)
        print('이번 실행 결과 저장:', saved_result)

print('고유 문서:', len(catalog), '/ 원본 파일 ID:', len(documents), '/ 청크:', len(chunks))
print('질의 검색 설정:')
show_json(config['retrieval'])
print('OpenAI 생성 설정(실행 옵션과 별도):')
show_json(config['generation'])
print('Phase 3 원문·청크·인덱스 버전 검사: 통과')


고유 문서: 99 / 원본 파일 ID: 100 / 청크: 18266
질의 검색 설정:
{
  "method": "bm25",
  "top_k": 8,
  "per_doc_top_k": 6,
  "fetch_k": 40,
  "mmr_lambda": 0.75,
  "max_context_bytes": 42000,
  "max_question_chars": 4000
}
OpenAI 생성 설정(실행 옵션과 별도):
{
  "enabled": false,
  "model": "gpt-5.4-mini",
  "max_output_tokens": 3000,
  "max_calls": 30,
  "max_request_bytes": 60000,
  "max_total_input_bytes": 600000,
  "cache_enabled": true,
  "reasoning_effort": "none"
}
Phase 3 원문·청크·인덱스 버전 검사: 통과


## 3. 공개 dev 문서에서 실습 대상 고르기

아래 실습은 dev의 질문 원문과 `context.selected_doc_ids`만 사용합니다.
정답 금액·날짜나 `target_doc_ids`를 검색어·문서 선택에 넣지 않습니다.
요약 실습은 dev 요약 사례 중 청크 수가 작은 문서를 골라 첫 실행 부담을 줄입니다.
이 선택은 실습 예제 선택이며 전체 dev 성능 측정을 대체하지 않습니다.

`PUBLIC_DOC_IDS`는 이번 API 실습에서 전송을 허용할 공개 문서 목록입니다.
실 API 옵션을 켜기 전에 표시된 문서명과 SHA를 확인하고 필요한 ID만 남기세요.
비공개 문서를 추가할 때는 전송 범위를 별도로 검토해야 합니다.


In [20]:
dev_cases = read_jsonl(ROOT / 'eval/questions.dev.jsonl')
assert all(case.get('split') == 'dev' for case in dev_cases)
demo_cases = {}
for feature in ['fact', 'comparison', 'followup', 'uncertainty']:
    demo_cases[feature] = next(case for case in dev_cases
                               if case.get('feature') == feature
                               and case.get('context', {}).get('selected_doc_ids'))
summary_candidates = [case for case in dev_cases
                      if case.get('feature') == 'summary'
                      and len(case.get('context', {}).get('selected_doc_ids', [])) == 1]
demo_cases['summary'] = min(summary_candidates,
    key=lambda case: len(index.get_document_chunks(case['context']['selected_doc_ids'][0])))
summary_doc_id = demo_cases['summary']['context']['selected_doc_ids'][0]

PUBLIC_DOC_IDS = sorted({doc_id for case in demo_cases.values()
                         for doc_id in case['context']['selected_doc_ids']})
approved_ids = list(index.resolve_doc_ids(PUBLIC_DOC_IDS))
allowed_sources = {doc_id: catalog_by_id[doc_id]['sha256'] for doc_id in approved_ids}
print('실습 사례:')
for feature, case in demo_cases.items():
    print(feature, case['case_id'], case.get('question', '')[:180])
print('API 사용을 허용할 공개 문서:')
for doc_id in approved_ids:
    print(doc_id, catalog_by_id[doc_id]['file_name'], allowed_sources[doc_id])
print('holdout 질문·정답: 읽지 않음')


실습 사례:
fact F001 2027년 글로벌 패션브랜드 육성 용역의 용역비는 얼마이며 부가세가 포함되나요?
comparison C001 선택한 다음 두 RFP를 비교해 줘: 「한국주택금융공사: HF 인사이트 시스템 개발」 / 「한국주택금융공사: 부동산 등기정보 시스템 구축」. 비교 항목은 예산(부가세 및 개발·HW 내역이 있으면 포함)과 사업기간(안정화·조정 조건 포함)야. 문서별 열을 두는 표로 정리하고 각 데이터 칸에 근거를 붙여 줘. 기간이나 예산의
followup D001 한국주택금융공사의 HF 인사이트 시스템 개발 예산이 얼마야? 세금 포함 여부도 알려줘.
uncertainty U003 이 AI 기반 관광 B2C 플랫폼 통합 ISP 수립 사업의 최종 낙찰업체와 실제 계약금액을 확정해서 알려줘.
summary S002 선택한 문서 전체를 검토하여 사업명, 발주기관, 목적, 주요 범위·필수요건, 예산과 VAT, 수행기간, 참가·수행자격, 입찰 제출서류, 제출기한, 제출방식, 평가기준의 11개 항목으로 요약해 주세요. 공고문 참조 사항과 확인되지 않는 사항도 구별해 주세요.
API 사용을 허용할 공개 문서:
161LuHCu_ZzUIDue5yGL_rRKMUe10FZ38 한국관광공사_2026_AI 기반 관광 B2C 플랫폼 통합 ISP 수립_제안요청서.hwp a0429ca4da02110586ecf5396ca89edf17c53dd26db9b26671f0802892791a8e
1Eq8gIdUnnLRbbEBiaU29wYji2YglNNuu 인천공항시설관리(주)_2026_2026년도 귀속 연말정산 위탁운영 용역_과업내용서.hwp 7a592ea538ed00d8dca697c8cab8fadd110ec2a15902f8235b80bb7d960027bc
1G2at8Un7y8JIn5awuVxqV7OOKIj7zT4B 한국주택금융공사_2023_HF 인사이트 시스템 개발_제안요청서.hwp 8c293ac2bb077438c0aaa974746d203a9196ec3eaa0d2c7971e5f6

## 4. API 호출 전에 실제 검색 근거 확인

먼저 원문에 어떤 근거가 검색되는지 확인합니다.
각 청크는 문서·원본 SHA·페이지/섹션 위치·원문 문자 범위를 유지합니다.
답변 생성기는 이 근거 안에서만 인용할 수 있습니다. 검색 점수는 답변 정확도 확률이 아닙니다.


In [22]:
fact_case = demo_cases['fact']
fact_ids = list(fact_case['context']['selected_doc_ids'])
retrieval_config = config['retrieval']
evidence_preview = index.search(fact_case['question'],
    top_k=retrieval_config.get('top_k', 8), allowed_doc_ids=fact_ids,
    method=retrieval_config.get('method', 'bm25'),
    fetch_k=retrieval_config.get('fetch_k', 40),
    mmr_lambda=retrieval_config.get('mmr_lambda', 0.75))
validated_preview = registry.validate_chunks(evidence_preview, allowed_doc_ids=fact_ids)
print('질문:', fact_case['question'])
print('검증한 검색 근거 수:', len(validated_preview))
for chunk in validated_preview[:3]:
    print('문서:', chunk['file_name'])
    print('청크:', chunk['chunk_id'], '/ 원문 위치:', chunk['locator'])
    print('원본 SHA:', chunk['sha256'])
    print(chunk['text'][:650])
    print()
assert all(chunk['doc_id'] in index.resolve_doc_ids(fact_ids) for chunk in validated_preview)
print('선택 문서 범위와 원문 연결 검사: 통과')


질문: 2027년 글로벌 패션브랜드 육성 용역의 용역비는 얼마이며 부가세가 포함되나요?
검증한 검색 근거 수: 8
문서: 서울특별시_2026_2027년 글로벌 패션브랜드 육성 용역_제안요청서.hwpx
청크: rfp:081719865dd99a38:2d6f15607efe3625752ac288 / 원문 위치: Contents/section0.xml / paragraph 64 / segment 1 … Contents/section0.xml / paragraph 77 / segment 1
원본 SHA: 081719865dd99a38ba822b39869efcf75b37add88f8d07741c8f8bf141060484
가. 과업 개요
ㅇ 과 업 명 : 2027년 글로벌 패션브랜드 육성 용역
ㅇ 과업기간 : 2026. 12. ~ 2027. 11. 중 프로그램 진행 기간 10개월
- 상 반 기 : ’26. 11. 30. ~ ’27. 4. 30.  (5개월)
- 하 반 기 : ’27. 7. 1. ~ ’27. 11. 30. (5개월)
ㅇ 용 역 비 : 650,000,000원(부가세 포함)
나. 과업목적
ㅇ 국내 유망 디자이너 브랜드의 글로벌 인지도 향상
ㅇ 현지 마케팅·세일즈 프로그램을 통해 해외 패션시장의 효율적 공략 지원
다. 과업범위 : 2027 아시아 패션 세일즈 프로그램 및 밀라노 패션위크 파트너십 프로그램 운영

문서: 서울특별시_2026_2027년 글로벌 패션브랜드 육성 용역_제안요청서.hwpx
청크: rfp:081719865dd99a38:7df0110c14f4e3a6cbb397a8 / 원문 위치: Contents/section0.xml / paragraph 1898 / segment 1 … Contents/section0.xml / paragraph 1943 / segment 1
원본 SHA: 081719865dd99a38ba822b39869efcf75b37add88f8d07741c8f8bf141060484
가 격 제 안 서
사 업 명
2027년 글로벌 패션브랜드

## 5. 허위 인용을 거르는지 검사

아래는 모델 응답이 아니라 **잘못된 인용을 넣은 검증용 입력**입니다.
원문에 없는 인용문은 거절되어야 합니다. 이 셀은 실제 답변을 생성하지 않습니다.
인용문의 정확한 일치는 출처를 확인하는 검사이며 주장의 의미적 정답 판정은 아닙니다.


In [23]:
if validated_preview:
    source_chunk = validated_preview[0]
    fabricated_quote = '검증용으로 삽입한 원문에 없는 인용 문자열 __PHASE4_INVALID_QUOTE__'
    assert fabricated_quote not in source_chunk['text']
    invalid_input = {'status': 'answered', 'claims': [{
        'doc_id': source_chunk['doc_id'], 'category': 'other',
        'text': '검증용 문장',
        'citations': [{'chunk_id': source_chunk['chunk_id'], 'quote': fabricated_quote}]}],
        'missing_information': []}
    rejected = registry.validate_answer(invalid_input, validated_preview, allowed_doc_ids=fact_ids)
    assert not rejected['valid']
    print('원문에 없는 인용 거절: 통과')
    print('검증 오류:')
    show_json(rejected['errors'])
    print('의미적 정답 검증 여부:', rejected['semantic_entailment_verified'])
else:
    print('검색 근거가 없어 허위 인용 실험을 생략했습니다. 답변 생성 단계에서도 근거 부족으로 처리해야 합니다.')


원문에 없는 인용 거절: 통과
검증 오류:
[
  "인용문이 청크 원문과 정확히 일치하지 않습니다."
]
의미적 정답 검증 여부: False


## 6. API 연결과 사용량 상한

`ENABLE_OPENAI=True`와 실행할 기능의 옵션을 함께 켰을 때만 API를 사용합니다.
키가 환경변수에 없으면 로컬 `getpass` 입력창으로 입력합니다. 키를 셀 소스나 출력에 넣지 않습니다.
선택한 질문과 승인한 원문 근거 텍스트가 OpenAI로 전송됩니다.

요청 수·요청별 입력 bytes·누적 입력 bytes·출력 토큰 상한을 적용합니다.
bytes는 정확한 모델 토큰 수나 청구 금액이 아닙니다. 실제 토큰 사용량은 응답 usage에 따로 기록합니다.
마지막 셀은 보고된 토큰 수와 설정 파일의 모델별 단가로 USD 추정 비용을 표시합니다.
실패한 요청의 사용량이 확인되지 않으면 추정이 불완전하다고 표시하고, 대역 토큰 사용량의 금액은 가정값으로 구분합니다.
같은 요청은 로컬 캐시를 사용할 수 있으며 캐시 적중은 새 API 호출과 구분합니다.
실습 뒤 코드를 수정해도 기존 결과의 출처·모델·프롬프트가 같은지 확인해야 합니다.


In [24]:
generation_options = dict(config['generation'])
generation_options['enabled'] = bool(ENABLE_OPENAI)
if ENABLE_OPENAI and any(live_options) and not os.environ.get('OPENAI_API_KEY'):
    from getpass import getpass
    entered_key = getpass('이 Mac에서 사용할 OPENAI_API_KEY: ').strip()
    if not entered_key:
        raise ValueError('API 키가 비어 있습니다. 실 API 옵션을 끄거나 로컬 환경변수에 키를 설정하세요.')
    os.environ['OPENAI_API_KEY'] = entered_key
    del entered_key

generation_llm = OpenAIGroundedLLM(ROOT / 'phase4_outputs/generation_cache',
    config=LLMConfig(**generation_options), allowed_sources=allowed_sources)
if ENABLE_OPENAI and any(live_options):
    from datetime import datetime, timezone
    from uuid import uuid4
    started_at = datetime.now(timezone.utc)
    run_id = started_at.strftime('%Y%m%dT%H%M%S_%fZ') + '_' + uuid4().hex[:8]
    RUN_OUTPUT_DIR = ROOT / 'phase4_outputs/runs' / run_id
    RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    RESULT_SEQUENCE = 0
    run_context = {
        'run_id': run_id, 'started_at_utc': started_at.isoformat(),
        'generation_options': generation_options,
        'phase4_config': {name: config[name] for name in
            ['version', 'generation', 'retrieval', 'summary', 'evaluation', 'semantic', 'pricing']
            if name in config},
        'index_fingerprint': index.get_metadata()['index_fingerprint'],
        'approved_sources': allowed_sources,
        'dependency_versions': runtime_dependency_versions,
        'python_executable': sys.executable, 'python_version': sys.version.split()[0],
        'prompt_version': PROMPT_VERSION,
        'run_options': {'RUN_LIVE_QA': RUN_LIVE_QA, 'RUN_LIVE_SUMMARY': RUN_LIVE_SUMMARY,
                        'RUN_DEV_EVALUATION': RUN_DEV_EVALUATION,
                        'RUN_SEMANTIC_COMPARISON': RUN_SEMANTIC_COMPARISON,
                        'DEV_CASE_LIMIT': DEV_CASE_LIMIT},
        'credentials_saved': False,
    }
    save_run_json('run_context', run_context)
    print('이번 실행 전용 출력 폴더:', RUN_OUTPUT_DIR)
print('생성 모델:', generation_options['model'])
print('실 API 활성화:', ENABLE_OPENAI)
print('API 키 설정 여부:', bool(os.environ.get('OPENAI_API_KEY')))
print('승인된 공개 문서 수:', len(allowed_sources))
print('실행 전 사용량:')
show_json(generation_llm.usage_report())


이번 실행 전용 출력 폴더: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase4_local_20261111/phase4_outputs/runs/20261111T084431_338782Z_d966ed59
생성 모델: gpt-5.4-mini
실 API 활성화: True
API 키 설정 여부: True
승인된 공개 문서 수: 5
실행 전 사용량:
{
  "api_calls": 0,
  "live_api_calls": 0,
  "mock_calls": 0,
  "cache_hits": 0,
  "failed_attempts": 0,
  "attempted_input_bytes": 0,
  "reserved_max_output_tokens": 0,
  "reported_input_tokens": 0,
  "reported_output_tokens": 0,
  "reported_cached_input_tokens": 0,
  "reported_total_tokens": 0,
  "usage_unknown_attempts": 0,
  "request_latency_ms": 0.0,
  "max_calls": 30,
  "max_total_input_bytes": 600000,
  "execution_mode": "live_openai",
  "enabled": true,
  "budget_unit": "UTF-8 serialized request bytes; not token count or currency"
}


## 7. 사실 Q&A와 출처 인용

`RUN_LIVE_QA=True`일 때 아래부터 Q&A·비교·후속 질문·근거 부족 예제를 실행합니다.
각 주장은 구조화된 출력으로 받고, 그 주장의 인용문이 이번 검색 근거와 원문에 존재하는지 검사합니다.
검증 실패 응답은 모델의 사실 문장을 정상 답변처럼 표시하지 않습니다.
`answered`여도 의미적으로 옳은지 별도 검토해야 합니다. 처리 오류와 근거 부족 상태도 구분합니다.


In [26]:
qa_result = None
if ENABLE_OPENAI and RUN_LIVE_QA:
    qa_session = AnswerSession(index, generation_llm, registry,
                              config=retrieval_config, active_doc_ids=fact_ids)
    qa_result = qa_session.ask(fact_case['question'], feature='qa')
    show_result(qa_result)
else:
    print('Q&A 생성 미실행: ENABLE_OPENAI=True, RUN_LIVE_QA=True일 때 실행합니다.')
    print('실제 질문:', fact_case['question'])
    print('현재는 원문 근거만 확인했습니다. 생성 답변 정답률은 측정하지 않았습니다.')


근거를 연결한 답변

- **예산**: 용역비는 650,000,000원이며, 부가세 포함입니다.
  - 출처: 서울특별시\_2026\_2027년 글로벌 패션브랜드 육성 용역\_제안요청서.hwpx · Contents/section0.xml / paragraph 64 / segment 1 … Contents/section0.xml / paragraph 77 / segment 1
    원문: “ㅇ 용 역 비 : 650,000,000원(부가세 포함)”

인용문과 원문 위치를 대조했습니다. 인용문이 주장 전체를 뒷받침하는지는 별도 검토가 필요합니다.
{
  "status": "answered",
  "execution_status": "completed",
  "answer_generated": true
}
이번 실행 결과 저장: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase4_local_20261111/phase4_outputs/runs/20261111T084431_338782Z_d966ed59/result_002.json


## 8. 문서별 근거를 확보한 비교

비교는 선택 문서마다 근거를 확보한 뒤 합칩니다.
한 문서의 높은 검색 점수 때문에 다른 문서의 근거가 빠지는 문제를 줄이기 위한 경로입니다.
예산의 단위·부가세 조건·기간의 기준은 인용문과 함께 확인해야 합니다.
문서 간 기준이 다르면 우열을 임의로 추천하지 않습니다.


In [27]:
comparison_case = demo_cases['comparison']
comparison_result = None
if ENABLE_OPENAI and RUN_LIVE_QA:
    comparison_session = AnswerSession(index, generation_llm, registry,
        config=retrieval_config, active_doc_ids=comparison_case['context']['selected_doc_ids'])
    comparison_result = comparison_session.ask(comparison_case['question'], feature='comparison')
    show_result(comparison_result)
else:
    print('비교 답변 미실행')
    print('질문:', comparison_case['question'])
    print('사용자 선택 문서:', comparison_case['context']['selected_doc_ids'])


생성 결과의 출처·형식 검사에 실패해 답변을 표시하지 않습니다.
{
  "status": "not_generated",
  "execution_status": "validation_failed",
  "answer_generated": false
}
이번 실행 결과 저장: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase4_local_20261111/phase4_outputs/runs/20261111T084431_338782Z_d966ed59/result_003.json


## 9. 후속 질문과 근거 부족 응답

대화에는 평가 정답이나 과거 assistant의 사실 문장을 새 근거로 넣지 않습니다.
사용자가 선택한 문서와 실제 사용자 발화만 유지하고 현재 질문으로 다시 검색합니다.
근거를 확보하지 못하면 확인 범위를 표시하며 문서 전체에 해당 조건이 없다고 단정하지 않습니다.
아래는 dev 대화의 첫 두 사용자 발화만 실행합니다.


In [9]:
followup_case = demo_cases['followup']
uncertainty_case = demo_cases['uncertainty']
followup_results = []
uncertainty_result = None
if ENABLE_OPENAI and RUN_LIVE_QA:
    conversation = AnswerSession(index, generation_llm, registry,
        config=retrieval_config, active_doc_ids=followup_case['context']['selected_doc_ids'])
    for turn in followup_case['turns'][:2]:
        print('사용자:', turn['user'])
        result = conversation.ask(turn['user'], feature='qa')
        followup_results.append(result)
        show_result(result)
        print()
    uncertainty_session = AnswerSession(index, generation_llm, registry,
        config=retrieval_config, active_doc_ids=uncertainty_case['context']['selected_doc_ids'])
    print('근거 부족 예제 질문:', uncertainty_case['question'])
    uncertainty_result = uncertainty_session.ask(uncertainty_case['question'], feature='qa')
    show_result(uncertainty_result)
else:
    print('대화·근거 부족 생성 미실행')
    print('사용자 발화 예시:', [turn['user'] for turn in followup_case['turns'][:2]])
    print('근거 부족 예제 질문:', uncertainty_case['question'])


사용자: 한국주택금융공사의 HF 인사이트 시스템 개발 예산이 얼마야? 세금 포함 여부도 알려줘.
답변 생성에 실패했습니다. API 설정·사용량 상한·모델 접근 권한을 확인하세요.
생성 실패 원인: OpenAI SDK is not installed in this notebook kernel; install requirements-phase4.txt
{
  "status": "not_generated",
  "execution_status": "error",
  "answer_generated": false
}
이번 실행 결과 저장: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase4_local_20261111/phase4_outputs/runs/20261111T083445_539924Z_dea8dfe0/result_003.json

사용자: 그건 몇 개월짜리야? 안정화도 그 안에 들어가?
답변 생성에 실패했습니다. API 설정·사용량 상한·모델 접근 권한을 확인하세요.
생성 실패 원인: OpenAI SDK is not installed in this notebook kernel; install requirements-phase4.txt
{
  "status": "not_generated",
  "execution_status": "error",
  "answer_generated": false
}
이번 실행 결과 저장: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase4_local_20261111/phase4_outputs/runs/20261111T083445_539924Z_dea8dfe0/result_004.json

근거 부족 예제 질문: 이 AI 기반 관광 B2C 플랫폼 통합 ISP 수립 사업의 최종 낙찰업체와 실제 계약금액을 확정해서 알려줘.
답변 생성에 실패했습니다. API 설정·사용량 상한·모델 접근 권한을 확인하세요.
생성 실

## 10. 전체 문서 요약의 실행 계획

일반 검색 top-k 대신 해당 문서의 모든 인덱싱된 청크를 순서대로 배치에 넣습니다.
배치 수와 전체 입력 상한을 먼저 확인하고, 모든 배치를 실행할 예산이 부족하면 시작 전에 멈춥니다.
입력이 큰 문서를 조용히 잘라서 전체 요약이라고 부르지 않습니다.
아래 계획 셀은 OpenAI를 비활성화한 상태에서도 실행할 수 있습니다.


In [10]:
summarizer = FullDocumentSummarizer(index, generation_llm, registry, config=config['summary'])
summary_plan = summarizer.plan(summary_doc_id)
print('요약 대상:', catalog_by_id[index.resolve_doc_ids([summary_doc_id])[0]]['file_name'])
show_json({key: value for key, value in summary_plan.items()
           if key not in {'batches', 'source_chunk_ids'}})
planned_ids = [chunk['chunk_id'] for batch in summary_plan['batches'] for chunk in batch['chunks']]
source_ids = [chunk['chunk_id'] for chunk in index.get_document_chunks(summary_doc_id)]
assert set(planned_ids) == set(source_ids)
assert len(planned_ids) == len(source_ids)
print('문서의 모든 인덱싱된 청크가 정확히 한 배치에 배정됨: 통과')
print('실제 API 호출 수: 0회 (계획만 생성)')


요약 대상: 인천공항시설관리(주)_2026_2026년도 귀속 연말정산 위탁운영 용역_과업내용서.hwp
{
  "doc_id": "1Eq8gIdUnnLRbbEBiaU29wYji2YglNNuu",
  "question": "이 RFP의 제공된 부분을 한국어로 항목별 정리해 주세요. 사업 개요, 예산과 부가세, 사업기간, 참가자격, 요구사항, 제출 방식과 기한, 평가 기준, 산출물, 계약상 유의사항을 살피세요. 각 사실에는 제공된 원문에서 정확히 복사한 인용문을 붙이세요. 이 입력은 전체 문서의 일부일 수 있으므로 여기서 찾지 못한 항목을 문서 전체에 없다고 단정하지 마세요. 원문에 쓰인 금액·단위·조건·연도 차이를 그대로 보존하세요.",
  "chunk_count": 18,
  "map_calls": 1,
  "total_request_bytes": 28129,
  "uncached_input_bytes_upper_bound": 28129,
  "llm_max_calls": 30,
  "llm_max_total_input_bytes": 600000,
  "whole_session_budget_preflight_required": true,
  "max_output_tokens_upper_bound": 3000,
  "byte_budget_is_token_count": false,
  "estimated_price": null,
  "max_map_calls": 30,
  "max_batch_bytes": 35000,
  "batch_max_bytes": 35000,
  "can_execute": true,
  "block_reasons": [],
  "plan_fingerprint": "d48063da8eb6ab05d345942a9055a9f404e1981c14331f11909d43ec5b7a0f2d",
  "api_calls": 0,
  "scope": "Every indexed chunk appears once, in source order. This is a

## 11. 전체 문서 항목별 요약 실행

`RUN_LIVE_SUMMARY=True`일 때 모든 배치에서 생성한 주장을 검증하고 중복을 정리합니다.
사업 개요·예산·기간·자격·요구사항·제출·평가·산출물·위험 항목에서 미확인 항목도 표시합니다.
일부 배치가 실패하거나 검증되지 않으면 전체 문서가 성공적으로 요약됐다고 표시하지 않습니다.

입력 청크 커버리지 100%는 문서 청크를 모두 처리했다는 뜻입니다.
모든 요구사항의 의미를 정확히 요약했다는 판정은 사람의 항목별 검토가 필요합니다.


In [11]:
summary_result = None
if ENABLE_OPENAI and RUN_LIVE_SUMMARY:
    summary_result = summarizer.summarize(summary_doc_id)
    show_result(summary_result)
else:
    print('전체 문서 요약 생성 미실행')
    print('예정 map 요청:', summary_plan['map_calls'], '/ 입력 청크:', summary_plan['chunk_count'])
    print('ENABLE_OPENAI=True, RUN_LIVE_SUMMARY=True로 설정한 뒤 계획과 상한을 확인하세요.')


전체 문서 요약 생성 미실행
예정 map 요청: 1 / 입력 청크: 18
ENABLE_OPENAI=True, RUN_LIVE_SUMMARY=True로 설정한 뒤 계획과 상한을 확인하세요.


## 12. 새 dev 실행과 사람 검토용 기록

이 셀은 실제 OpenAI를 켠 경우에만 선택한 공개 dev 예제를 평가합니다.
실행 결과는 이번 실험의 `RUN_OUTPUT_DIR/dev` 폴더에 기록하고 Phase 3 보고서는 수정하지 않습니다.
같은 셀을 다시 실행하면 번호가 붙은 새 dev 폴더를 사용해 사람 검토 기록을 보존합니다.
같은 프롬프트·모델·설정에서 캐시를 사용한 요청과 새 API 호출을 구분합니다.

자동 지표는 상태, 인용문과 원문 위치의 유효성, 정답 근거 구간 회수 등을 측정합니다.
숫자·문자열 단서 일치는 참고 지표이며 의미적 정답률이 아닙니다.
생성 문장이 인용문에 의해 뒷받침되는지, 질문에 충분히 답했는지는 사람 검토 표에 기록합니다.
이전에 본 holdout은 Phase 4 설정 조정과 최종 검증에 다시 사용하지 않습니다.


In [12]:
dev_generation_report = None
if ENABLE_OPENAI and RUN_DEV_EVALUATION:
    from rfp_phase4.evaluation import run_dev_evaluation
    evaluation_cases = [demo_cases[feature] for feature in
                        ['fact', 'summary', 'comparison', 'followup', 'uncertainty']]
    for case in evaluation_cases:
        assert set(index.resolve_doc_ids(case['context']['selected_doc_ids'])).issubset(set(approved_ids))
    dev_alignment = json.loads((ROOT / 'reports/evidence_chunk_alignment.json').read_text(encoding='utf-8'))
    dev_output_dir = new_run_subdirectory('dev')
    dev_generation_report = run_dev_evaluation(
        evaluation_cases,
        answer_session_factory=lambda active_doc_ids=None: AnswerSession(
            index, generation_llm, registry, config=retrieval_config, active_doc_ids=active_doc_ids),
        summary_factory=lambda: FullDocumentSummarizer(
            index, generation_llm, registry, config=config['summary']),
        output_dir=dev_output_dir, max_cases=DEV_CASE_LIMIT,
        allow_live=True, alignment=dev_alignment, source_documents=documents, registry=registry,
        allowed_doc_ids=approved_ids)
    show_json({key: value for key, value in dev_generation_report.items()
               if key not in {'case_results', 'human_review_rows', 'results'}})
else:
    print('OpenAI dev 5기능 평가 미실행')
    print('기본 Run All의 통과 항목은 코드·원문 연결 검사이며 생성 품질 측정이 아닙니다.')


OpenAI dev 5기능 평가 미실행
기본 Run All의 통과 항목은 코드·원문 연결 검사이며 생성 품질 측정이 아닙니다.


## 13. 선택 실험: OpenAI 임베딩과 BM25를 같은 범위에서 비교

Chroma 선택 의존성이 필요하면 README의 별도 설치 안내를 따릅니다.
먼저 공개 승인 문서의 임베딩 계획을 출력하고, 선택 옵션을 켠 경우에만 구축·검색합니다.
첫 실험의 `SEMANTIC_DOC_IDS`는 사실 Q&A 문서 한 개로 제한합니다. 늘릴 때는 계획과 사용량 상한을 다시 확인하세요.
일부 문서만 색인하면 동일 문서 범위에서 BM25와 비교하고, 범위 밖 dev 사례는 제외 건수를 기록합니다.
결과를 전체 99개 문서의 의미 검색 성능으로 일반화하지 않습니다.


In [13]:
semantic_comparison = None
SEMANTIC_DOC_IDS = list(fact_ids)  # 첫 실험은 한 문서. 공개 승인 목록 안에서 명시적으로 조정.
if ENABLE_OPENAI and RUN_SEMANTIC_COMPARISON:
    from rfp_phase4.semantic import plan_semantic_index, build_semantic_index, compare_semantic_on_dev
    semantic_config = dict(config['semantic'])
    semantic_config['enabled'] = True
    semantic_plan = plan_semantic_index(index, SEMANTIC_DOC_IDS, config=semantic_config)
    show_json({key: value for key, value in semantic_plan.items()
               if key not in {'chunks', 'allowed_sources'}})
    semantic_directory = ROOT / 'phase4_outputs/semantic_indexes' / semantic_plan['corpus_fingerprint'][:12]
    semantic_index, semantic_build = build_semantic_index(
        index, SEMANTIC_DOC_IDS, semantic_directory,
        config=semantic_config, registry=registry)
    alignment = json.loads((ROOT / 'reports/evidence_chunk_alignment.json').read_text(encoding='utf-8'))
    semantic_comparison = compare_semantic_on_dev(
        index, semantic_index, dev_cases, alignment,
        top_ks=(5, 10, 20), max_cases=10, scope_mode='context')
    save_run_json('semantic_run_context', {'semantic_options': semantic_config,
        'selected_doc_ids': SEMANTIC_DOC_IDS,
        'corpus_fingerprint': semantic_plan['corpus_fingerprint'],
        'semantic_index_directory': str(semantic_directory), 'build_report': semantic_build})
    semantic_output = save_run_json('semantic_comparison', semantic_comparison)
    show_json({key: value for key, value in semantic_comparison.items()
               if key not in {'case_results', 'retrieval_reports', 'reports'}})
    print('의미 검색 비교 결과:', semantic_output)
else:
    print('OpenAI 임베딩 구축 및 dev 검색 비교: 미실행')


OpenAI 임베딩 구축 및 dev 검색 비교: 미실행


## 14. 실행 상태 확인

아래 통과 항목은 현재 실행에서 확인한 범위만 표시합니다.
실제 API를 끈 상태에서는 생성 답변·요약·의미 검색 성능을 측정했다고 표시하지 않습니다.
실무 완성 판정은 답변과 인용의 의미 검토, 항목 누락, 근거 부족 처리, 비용과 지연시간을 함께 봐야 합니다.


In [14]:
checks = {
    'notebook_dependency_versions_match_index': not version_mismatches,
    'phase3_sources_and_index_loaded': len(catalog) > 0 and len(chunks) > 0,
    'dev_only_examples_used': all(case['split'] == 'dev' for case in demo_cases.values()),
    'retrieved_source_chunks_validated': bool(validated_preview),
    'fabricated_quote_rejected': not rejected['valid'] if validated_preview else None,
    'full_document_plan_preserves_all_chunks': set(planned_ids) == set(source_ids) and len(planned_ids) == len(source_ids),
}
for name, passed in checks.items():
    print(name, ':', '통과' if passed is True else ('미실행' if passed is None else '실패'))
assert all(value is not False for value in checks.values())
print('Q&A 실행 경로 진입:', qa_result is not None)
print('전체 문서 요약 실행 경로 진입:', summary_result is not None)
for label, result in [('Q&A', qa_result), ('전체 문서 요약', summary_result)]:
    if result is not None:
        print(label, '/ status:', result.get('status'), '/ execution_status:', result.get('execution_status'))
print('dev 평가 실행 경로 진입:', dev_generation_report is not None)
print('dev 평가의 실제 API 실행 기록:',
      bool(dev_generation_report and dev_generation_report.get('live_api_executed')))
print('의미 검색 비교 실행 경로 진입:', semantic_comparison is not None)
if semantic_comparison is not None:
    print('의미 검색 비교 모드:', semantic_comparison.get('execution_mode', '비교 가능한 사례 없음'))
    print('의미 검색 임베딩 호출·캐시 기록:')
    show_json(semantic_index.usage_report())
print('실제 사용량 / 캐시 기록:')
final_usage = generation_llm.usage_report()
print('생성의 실제 API 호출 시도 수:', final_usage.get('live_api_calls', 0))
print('생성의 로컬 캐시 적중 수:', final_usage.get('cache_hits', 0))
show_json(final_usage)
if generation_options['model'] == config['pricing']['model']:
    print('보고된 토큰 기준 USD 비용 추정:')
    final_cost_estimate = estimate_usage_cost(final_usage, config['pricing'])
    show_json(final_cost_estimate)
    print('비용은 설정된 단가에 따른 추정이며 실제 청구서가 아닙니다.')
    print('미확인 실패 요청 사용량·세금·추가 요금은 포함하지 않습니다. 대역 사용량의 금액은 가정값입니다.')
else:
    final_cost_estimate = None
    print('생성 모델과 단가 설정의 모델이 달라 비용 추정을 생략합니다. configs/phase4.json의 pricing을 맞추세요.')
print('출처 인용 유효성과 의미적 정답은 별도로 검토합니다.')
if RUN_OUTPUT_DIR is not None:
    saved_run_summary = save_run_json('run_summary', {
        'generation_usage': final_usage, 'generation_cost_estimate': final_cost_estimate,
        'qa_status': qa_result.get('status') if qa_result else None,
        'qa_execution_status': qa_result.get('execution_status') if qa_result else None,
        'summary_status': summary_result.get('status') if summary_result else None,
        'dev_live_api_executed': bool(dev_generation_report and dev_generation_report.get('live_api_executed')),
        'semantic_usage': semantic_index.usage_report() if semantic_comparison is not None else None,
        'semantic_answer_quality_verified': False, 'credentials_saved': False})
    print('이번 실행의 사용량·비용·상태 기록:', saved_run_summary)
    print('이번 실행 전용 출력 폴더:', RUN_OUTPUT_DIR)
else:
    print('기본 오프라인 실행: 새 API 실험 폴더를 만들지 않았습니다.')
print('Phase 4 로컬 출력 기준 폴더:', ROOT / 'phase4_outputs')


notebook_dependency_versions_match_index : 통과
phase3_sources_and_index_loaded : 통과
dev_only_examples_used : 통과
retrieved_source_chunks_validated : 통과
fabricated_quote_rejected : 통과
full_document_plan_preserves_all_chunks : 통과
Q&A 실행 경로 진입: True
전체 문서 요약 실행 경로 진입: False
Q&A / status: not_generated / execution_status: error
dev 평가 실행 경로 진입: False
dev 평가의 실제 API 실행 기록: False
의미 검색 비교 실행 경로 진입: False
실제 사용량 / 캐시 기록:
생성의 실제 API 호출 시도 수: 0
생성의 로컬 캐시 적중 수: 0
{
  "api_calls": 0,
  "live_api_calls": 0,
  "mock_calls": 0,
  "cache_hits": 0,
  "failed_attempts": 0,
  "attempted_input_bytes": 0,
  "reserved_max_output_tokens": 0,
  "reported_input_tokens": 0,
  "reported_output_tokens": 0,
  "reported_cached_input_tokens": 0,
  "reported_total_tokens": 0,
  "usage_unknown_attempts": 0,
  "request_latency_ms": 0.0,
  "max_calls": 30,
  "max_total_input_bytes": 600000,
  "execution_mode": "live_openai",
  "enabled": true,
  "budget_unit": "UTF-8 serialized request bytes; not token count or currency"